# Map 활용 및 Encapsulation 개념 관련 실습

- 이름: 김도연
- 학번: 20252183

---

In [1]:
// 학생 레코드의 구성 요소는 학번과 이름
record Student(int id, String name) { }

In [5]:
import java.util.*;

class School {
    private final String name;
    private final HashMap<Integer, Student> students;

    // students를 빈 가변 맵으로 초기화시키는 생성자
    School(String name) {
        if (name == null) {
            throw new IllegalArgumentException("학교 이름은 null일 수 없습니다.");
        }
        this.name = name;
        this.students = new HashMap<>();
    }

    // add or update a student
    public void update(Student s) {
        if (s == null) {
            throw new IllegalArgumentException("학생 객체는 null일 수 없습니다.");
        }
        // 아직 등록된 학번이 아니면 새로 추가하고, 이미 등록되어 있으면 업데이트
        students.put(s.id(), s);
    }

    // 학교 이름에 대한 참조를 리턴하는 메소드
    public String getName() {
        return name;
    }

    // 학생 Map에 대한 참조를 리턴하는 메소드 (캡슐화 침해 원인)
    public Map<Integer, Student> getStudents() {
        return students;
    }

    @Override
    public boolean equals(Object o) {
        if (this == o) return true;
        if (o == null || getClass() != o.getClass()) return false;
        School school = (School) o;
        return Objects.equals(name, school.name) && Objects.equals(students, school.students);
    }

    @Override
    public int hashCode() {
        return Objects.hash(name, students);
    }

    @Override
    public String toString() {
        return "School{name='" + name + "', students=" + students + "}";
    }
}

주석에 안내된 대로 School 클래스의 생성자와 update 메소드, hascCode, equals, toString 메소드들을 적절히 작성한 다음,
현재의 School 클래스가 설계가 Encapsulation을 만족하지 못하는 예시를 아래에 작성해 보라.

In [6]:
School school = new School("한남대학교");
school.update(new Student(20260001, "김철수"));

System.out.println("변경 전: " + school.getStudents());

// getStudents()가 내부 참조를 그대로 반환하므로 외부에서 내부 Map을 직접 조작 가능
Map<Integer, Student> externalMap = school.getStudents();
externalMap.put(99999999, new Student(99999999, "외부침입자"));

// School 내부의 데이터가 외부 조작으로 변경됨
System.out.println("변경 후: " + school.getStudents());

변경 전: {20260001=Student[id=20260001, name=김철수]}
변경 후: {20260001=Student[id=20260001, name=김철수], 99999999=Student[id=99999999, name=외부침입자]}


School2클래스를 School과 똑같은 내용이되 메소드 하나만 다르게 바꿔 작성해서 캡슐화 원리가 지켜지도록 수정해 보라

In [7]:
import java.util.*;

class School2 {
    private final String name;
    private final HashMap<Integer, Student> students;

    School2(String name) {
        if (name == null) {
            throw new IllegalArgumentException("학교 이름은 null일 수 없습니다.");
        }
        this.name = name;
        this.students = new HashMap<>();
    }

    public void update(Student s) {
        if (s == null) {
            throw new IllegalArgumentException("학생 객체는 null일 수 없습니다.");
        }
        students.put(s.id(), s);
    }

    public String getName() {
        return name;
    }

    // 캡슐화 보장을 위해 수정 불가능한 unmodifiableMap 리턴
    public Map<Integer, Student> getStudents() {
        return Collections.unmodifiableMap(students);
    }

    @Override
    public boolean equals(Object o) {
        if (this == o) return true;
        if (o == null || getClass() != o.getClass()) return false;
        School2 school2 = (School2) o;
        return Objects.equals(name, school2.name) && Objects.equals(students, school2.students);
    }

    @Override
    public int hashCode() {
        return Objects.hash(name, students);
    }

    @Override
    public String toString() {
        return "School2{name='" + name + "', students=" + students + "}";
    }
}

School에서 Encapsulation이 깨졌던 예시와 같은 내용의 School2에 대한 예시를 작성해서 이번에는 Encapsulation 원리가 지켜진다는 것을 보이라. (Error가 나는 방식으로 했다면 Error나는 실행결과 그대로를 제출. Error가 나지 않는 방힉으로 했다면 Error없이 실행되는 실행결과 그대로를 제출)

In [8]:
School2 school2 = new School2("한남대학교");
school2.update(new Student(20260001, "김철수"));

System.out.println("초기 상태: " + school2.getStudents());

try {
    Map<Integer, Student> externalMap = school2.getStudents();
    // 수정 시도 -> UnsupportedOperationException 발생
    externalMap.put(99999999, new Student(99999999, "외부침입자"));
} catch (UnsupportedOperationException e) {
    System.out.println("예외 발생: 캡슐화가 정상적으로 지켜졌습니다! (" + e.getClass().getName() + ")");
}

System.out.println("최종 상태 (변경 없음): " + school2.getStudents());

초기 상태: {20260001=Student[id=20260001, name=김철수]}
예외 발생: 캡슐화가 정상적으로 지켜졌습니다! (java.lang.UnsupportedOperationException)
최종 상태 (변경 없음): {20260001=Student[id=20260001, name=김철수]}
